# Probabilistic Learning — Illustrative Notebook

This notebook accompanies a lecture outline based on **Chapter 20: Learning Probabilistic Models** from *Artificial Intelligence: A Modern Approach*.

It is designed for:
- **Google Colab**
- **Local Jupyter environments**
- **Python 3.12+**

Assumptions:
- `numpy` and `matplotlib` are already installed

The notebook is organized into short sections. Each section begins with a markdown explanation and is followed by one or more runnable code cells.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)


## 1. Statistical Learning Framework — Bayesian Updating over Hypotheses

The first example reproduces the chapter's **candy-bag learning scenario**.  
We assume there are several possible bag types, each with a different proportion of lime candies.

We:
1. start with a prior over hypotheses,
2. observe a sequence of candies,
3. update the posterior distribution over hypotheses,
4. compute the predictive probability that the next candy is lime.


In [ ]:
# Hypotheses: proportion of lime candies in each bag
hypotheses = {
    "h1": 0.0,   # 0% lime
    "h2": 0.25,  # 25% lime
    "h3": 0.50,  # 50% lime
    "h4": 0.75,  # 75% lime
    "h5": 1.0    # 100% lime
}

# Prior from the chapter
prior = {
    "h1": 0.1,
    "h2": 0.2,
    "h3": 0.4,
    "h4": 0.2,
    "h5": 0.1
}

observations = ["lime"] * 10  # 10 lime candies in a row

def likelihood(obs, p_lime):
    return p_lime if obs == "lime" else (1 - p_lime)

def normalize(dist):
    total = sum(dist.values())
    return {k: v / total for k, v in dist.items()}

posterior = prior.copy()
history = [posterior.copy()]

for obs in observations:
    posterior = {
        h: posterior[h] * likelihood(obs, p_lime)
        for h, p_lime in hypotheses.items()
    }
    posterior = normalize(posterior)
    history.append(posterior.copy())

print("Final posterior after 10 lime candies:")
for h, p in posterior.items():
    print(f"{h}: {p:.6f}")

# Bayesian prediction for next candy being lime
p_next_lime = sum(hypotheses[h] * posterior[h] for h in hypotheses)
print(f"\nP(next candy is lime | data) = {p_next_lime:.6f}")


### Posterior and Predictive Curves

This plot shows two things:
- how belief in each hypothesis changes as evidence accumulates,
- how the predictive probability for the next candy evolves.


In [ ]:
steps = list(range(len(history)))
posterior_matrix = {h: [hist[h] for hist in history] for h in hypotheses}
predictions = [
    sum(hypotheses[h] * hist[h] for h in hypotheses)
    for hist in history
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for h in hypotheses:
    axes[0].plot(steps, posterior_matrix[h], marker="o", label=h)
axes[0].set_title("Posterior probability of each hypothesis")
axes[0].set_xlabel("Number of observations")
axes[0].set_ylabel("Posterior probability")
axes[0].legend()

axes[1].plot(steps, predictions, marker="o")
axes[1].set_title("Bayesian prediction: P(next candy is lime)")
axes[1].set_xlabel("Number of observations")
axes[1].set_ylabel("Probability")

plt.tight_layout()
plt.show()


## 2. Simplified Learning Methods — MAP and ML

Exact Bayesian prediction uses *all* hypotheses.  
Two common simplifications are:

- **MAP**: use the single most probable posterior hypothesis,
- **ML**: use the single hypothesis with highest likelihood.

The following cell compares all three approaches on the same candy example.


In [ ]:
# MAP hypothesis
h_map = max(posterior, key=posterior.get)
p_map_next_lime = hypotheses[h_map]

# ML hypothesis: choose hypothesis with largest likelihood only
def full_likelihood(observations, p_lime):
    if p_lime == 0.0 and "lime" in observations:
        return 0.0
    if p_lime == 1.0 and "cherry" in observations:
        return 0.0
    prod = 1.0
    for obs in observations:
        prod *= likelihood(obs, p_lime)
    return prod

likelihoods = {h: full_likelihood(observations, p_lime) for h, p_lime in hypotheses.items()}
h_ml = max(likelihoods, key=likelihoods.get)
p_ml_next_lime = hypotheses[h_ml]

print(f"Bayesian prediction P(next lime | data): {p_next_lime:.4f}")
print(f"MAP hypothesis: {h_map}, predicts next-lime probability {p_map_next_lime:.4f}")
print(f"ML hypothesis:  {h_ml}, predicts next-lime probability {p_ml_next_lime:.4f}")


### MDL / Occam-style tradeoff

The chapter also interprets model choice as a balance between:
- **good fit to the data**, and
- **model simplicity**.

The short toy example below illustrates a Minimum Description Length (MDL) style comparison.


In [ ]:
# Toy example: three hypotheses with fit (negative log-likelihood) and complexity cost
models = {
    "simple_model": {"nll": 12.0, "complexity_bits": 2.0},
    "medium_model": {"nll": 9.0, "complexity_bits": 5.0},
    "complex_model": {"nll": 6.5, "complexity_bits": 10.0},
}

print("Model comparison by MDL score = data encoding cost + model encoding cost\n")
for name, vals in models.items():
    mdl = vals["nll"] + vals["complexity_bits"]
    print(f"{name:14s}  NLL={vals['nll']:5.1f}  complexity={vals['complexity_bits']:5.1f}  MDL={mdl:5.1f}")

best = min(models, key=lambda m: models[m]["nll"] + models[m]["complexity_bits"])
print(f"\nChosen by MDL: {best}")


## 3. Learning with Complete Data — Maximum Likelihood for a Bernoulli Parameter

Suppose the proportion of cherry candies is unknown and must be estimated from observed data.

For a Bernoulli-style model,
- the parameter is \( \theta = P(\text{cherry}) \),
- the maximum-likelihood estimate is simply the observed fraction of cherries.


In [ ]:
observations = ["cherry", "lime", "cherry", "cherry", "lime", "cherry", "cherry", "cherry"]
n = len(observations)
c = observations.count("cherry")
l = observations.count("lime")

theta_ml = c / n

print(f"Observed cherries: {c}")
print(f"Observed limes:    {l}")
print(f"ML estimate theta = P(cherry) = {theta_ml:.4f}")


### Log-likelihood curve

This cell plots the Bernoulli log-likelihood as a function of \(\theta\).  
Students can visually confirm that the maximum occurs at the observed frequency.


In [ ]:
thetas = np.linspace(0.001, 0.999, 500)
log_likelihoods = c * np.log(thetas) + l * np.log(1 - thetas)

theta_best = thetas[np.argmax(log_likelihoods)]

plt.figure(figsize=(7, 4))
plt.plot(thetas, log_likelihoods)
plt.axvline(theta_ml, linestyle="--", label=f"ML = {theta_ml:.3f}")
plt.axvline(theta_best, linestyle=":", label=f"grid max = {theta_best:.3f}")
plt.xlabel("theta = P(cherry)")
plt.ylabel("log likelihood")
plt.title("Bernoulli log-likelihood")
plt.legend()
plt.show()


## 3.1 Learning Parameters in a Bayesian Network from Complete Data

With **complete data**, learning the parameters of a Bayesian network is often straightforward:
we estimate each conditional probability table entry by **counting frequencies**.

The example below uses a tiny network:

**Flavor → Wrapper**


In [ ]:
# Data: (Flavor, Wrapper)
data = [
    ("cherry", "red"),
    ("cherry", "red"),
    ("cherry", "green"),
    ("lime", "green"),
    ("lime", "red"),
    ("lime", "green"),
    ("cherry", "red"),
    ("lime", "green"),
]

# Estimate P(Flavor=cherry)
count_cherry = sum(1 for flavor, _ in data if flavor == "cherry")
theta = count_cherry / len(data)

# Estimate P(Wrapper=red | Flavor=cherry)
cherry_rows = [wrapper for flavor, wrapper in data if flavor == "cherry"]
theta1 = sum(1 for w in cherry_rows if w == "red") / len(cherry_rows)

# Estimate P(Wrapper=red | Flavor=lime)
lime_rows = [wrapper for flavor, wrapper in data if flavor == "lime"]
theta2 = sum(1 for w in lime_rows if w == "red") / len(lime_rows)

print(f"P(Flavor=cherry) ≈ {theta:.4f}")
print(f"P(Wrapper=red | Flavor=cherry) ≈ {theta1:.4f}")
print(f"P(Wrapper=red | Flavor=lime)   ≈ {theta2:.4f}")


## 4. Naive Bayes Model — Training and Prediction

Naive Bayes is a simple but important **generative classifier**.  
It assumes that the features are conditionally independent given the class.

The example below:
- trains a binary naive Bayes classifier from scratch,
- applies Laplace smoothing,
- predicts the class probability for one new example.


In [ ]:
# Toy dataset: [study_hard, slept_well, attended_class] -> pass_exam
X = np.array([
    [1, 1, 1],
    [1, 0, 1],
    [1, 1, 0],
    [0, 1, 1],
    [0, 0, 1],
    [0, 1, 0],
    [1, 0, 0],
    [0, 0, 0],
], dtype=int)

y = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=int)

# Laplace smoothing
alpha = 1.0

p_y1 = (np.sum(y == 1) + alpha) / (len(y) + 2 * alpha)
p_y0 = 1 - p_y1

# P(X_i=1 | Y=1) and P(X_i=1 | Y=0)
p_x1_given_y1 = (np.sum(X[y == 1], axis=0) + alpha) / (np.sum(y == 1) + 2 * alpha)
p_x1_given_y0 = (np.sum(X[y == 0], axis=0) + alpha) / (np.sum(y == 0) + 2 * alpha)

def predict_proba_naive_bayes(x):
    x = np.asarray(x)
    log_p1 = math.log(p_y1)
    log_p0 = math.log(p_y0)

    for i, xi in enumerate(x):
        if xi == 1:
            log_p1 += math.log(p_x1_given_y1[i])
            log_p0 += math.log(p_x1_given_y0[i])
        else:
            log_p1 += math.log(1 - p_x1_given_y1[i])
            log_p0 += math.log(1 - p_x1_given_y0[i])

    m = max(log_p0, log_p1)
    p0 = math.exp(log_p0 - m)
    p1 = math.exp(log_p1 - m)
    total = p0 + p1
    return np.array([p0 / total, p1 / total])

x_new = np.array([1, 1, 0])  # studies, sleeps, but missed class
proba = predict_proba_naive_bayes(x_new)

print("P(Y=0 | x), P(Y=1 | x) =", proba)
print("Predicted class =", int(np.argmax(proba)))


## 5. Generative vs Discriminative Models

The chapter briefly distinguishes between:

- **generative models**, which learn something like \(P(X, Y)\),
- **discriminative models**, which learn \(P(Y \mid X)\) directly.

The next cell does not build a full classifier. Instead, it gives a small numeric example of how a generative model can recover a posterior using Bayes' rule.


In [ ]:
# Generative viewpoint: estimate P(X|Y) and P(Y), then infer P(Y|X)
# Discriminative viewpoint: model P(Y|X) directly

print("Generative model example:")
print("  Learn P(Y) and P(X|Y), then compute P(Y|X) via Bayes rule.")

print("\nDiscriminative model example:")
print("  Learn P(Y|X) directly, e.g. with logistic regression.")

# A tiny numerical example
p_y1 = 0.5
p_x_given_y1 = 0.8
p_x_given_y0 = 0.3

numerator = p_x_given_y1 * p_y1
denominator = numerator + p_x_given_y0 * (1 - p_y1)
p_y1_given_x = numerator / denominator

print(f"\nIf P(Y=1)=0.5, P(X=1|Y=1)=0.8, P(X=1|Y=0)=0.3, then:")
print(f"P(Y=1|X=1) = {p_y1_given_x:.4f}")


## 6. Continuous Parameter Learning — Gaussian Mean and Variance

For continuous data, maximum likelihood can be used to estimate the parameters of a Gaussian:
- mean \(\mu\),
- standard deviation \(\sigma\).

This example generates synthetic Gaussian data and estimates those parameters.


In [ ]:
data = rng.normal(loc=5.0, scale=2.0, size=200)

mu_ml = np.mean(data)
sigma_ml = np.sqrt(np.mean((data - mu_ml) ** 2))

print(f"Estimated mean (ML): {mu_ml:.4f}")
print(f"Estimated std  (ML): {sigma_ml:.4f}")

plt.figure(figsize=(7, 4))
plt.hist(data, bins=20, density=True, alpha=0.7)
plt.axvline(mu_ml, linestyle="--", label=f"mean = {mu_ml:.2f}")
plt.title("Sample from a Gaussian distribution")
plt.legend()
plt.show()


### Linear Gaussian Model / Linear Regression

The chapter shows that linear regression can be interpreted probabilistically:
if the output is generated by a line plus Gaussian noise,
then least squares corresponds to **maximum likelihood estimation**.

The next cell demonstrates this on synthetic data.


In [ ]:
# Generate synthetic linear data
x = np.linspace(0, 10, 50)
true_slope = 2.0
true_intercept = 1.0
noise = rng.normal(0, 1.0, size=len(x))
y = true_slope * x + true_intercept + noise

# Closed-form least squares fit
X_design = np.column_stack([x, np.ones_like(x)])
theta_hat, *_ = np.linalg.lstsq(X_design, y, rcond=None)
slope_hat, intercept_hat = theta_hat

print(f"Estimated slope:     {slope_hat:.4f}")
print(f"Estimated intercept: {intercept_hat:.4f}")

plt.figure(figsize=(7, 4))
plt.scatter(x, y, label="data")
plt.plot(x, true_slope * x + true_intercept, label="true line")
plt.plot(x, slope_hat * x + intercept_hat, label="fitted line")
plt.title("Linear regression as ML with Gaussian noise")
plt.legend()
plt.show()


## 7. Bayesian Parameter Learning — Beta Prior and Posterior

Maximum likelihood can behave badly on very small datasets.  
Bayesian parameter learning addresses this by placing a prior over the parameter itself.

For a Bernoulli-style probability \(\theta\), a standard conjugate prior is the **Beta distribution**.


In [ ]:
# Prior Beta(a, b)
a, b = 2, 2  # mildly informative prior centered at 0.5

observations = ["cherry", "lime", "cherry", "cherry", "lime", "cherry"]

for obs in observations:
    if obs == "cherry":
        a += 1
    else:
        b += 1

posterior_mean = a / (a + b)

print(f"Posterior Beta parameters: a={a}, b={b}")
print(f"Posterior mean of theta = P(cherry): {posterior_mean:.4f}")


### Visualizing Beta prior and posterior

This plot shows how a Beta prior is updated into a posterior after observing data.


In [ ]:
def beta_unnormalized(theta, a, b):
    return np.where(
        (theta >= 0) & (theta <= 1),
        theta ** (a - 1) * (1 - theta) ** (b - 1),
        0.0
    )

theta_grid = np.linspace(0.001, 0.999, 500)

pri_a, pri_b = 2, 2
post_a, post_b = a, b

prior_curve = beta_unnormalized(theta_grid, pri_a, pri_b)
post_curve = beta_unnormalized(theta_grid, post_a, post_b)

# Normalize numerically for plotting
prior_curve /= np.trapezoid(prior_curve, theta_grid)
post_curve /= np.trapezoid(post_curve, theta_grid)

plt.figure(figsize=(7, 4))
plt.plot(theta_grid, prior_curve, label=f"Prior Beta({pri_a},{pri_b})")
plt.plot(theta_grid, post_curve, label=f"Posterior Beta({post_a},{post_b})")
plt.xlabel("theta = P(cherry)")
plt.ylabel("density")
plt.title("Bayesian updating with a Beta prior")
plt.legend()
plt.show()


## 8. Nonparametric Density Estimation — k-NN Density

Nonparametric methods do not assume a fixed parametric form for the distribution.

The first example below implements a simple **1D k-nearest-neighbor density estimate**.


In [ ]:
samples = np.concatenate([
    rng.normal(-2, 0.4, size=80),
    rng.normal(1.5, 0.3, size=120)
])

def knn_density_1d(x_query, samples, k=10):
    distances = np.abs(samples - x_query)
    sorted_distances = np.sort(distances)
    radius = sorted_distances[k - 1]
    volume = 2 * radius  # 1D interval length
    return k / (len(samples) * volume) if volume > 0 else float("inf")

grid = np.linspace(-4, 4, 300)
density_vals = np.array([knn_density_1d(xq, samples, k=10) for xq in grid])

plt.figure(figsize=(7, 4))
plt.hist(samples, bins=30, density=True, alpha=0.4, label="histogram")
plt.plot(grid, density_vals, label="k-NN density (k=10)")
plt.title("1D k-nearest-neighbor density estimation")
plt.legend()
plt.show()


### Kernel Density Estimation

A second nonparametric method is **kernel density estimation (KDE)**.  
Each data point contributes a small smooth bump, and the final estimate is their average.


In [ ]:
def gaussian_kernel(u):
    return np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)

def kde_1d(x_query, samples, bandwidth=0.25):
    u = (x_query - samples) / bandwidth
    return np.mean(gaussian_kernel(u)) / bandwidth

grid = np.linspace(-4, 4, 300)
kde_vals = np.array([kde_1d(xq, samples, bandwidth=0.25) for xq in grid])

plt.figure(figsize=(7, 4))
plt.hist(samples, bins=30, density=True, alpha=0.4, label="histogram")
plt.plot(grid, kde_vals, label="Gaussian KDE")
plt.title("1D kernel density estimation")
plt.legend()
plt.show()


## 9. Hidden Variables — Why Latent Structure Helps

Many models include **hidden (latent) variables** that are not directly observed.
These hidden variables can capture underlying causes or group membership.

The following cell simulates a dataset produced by two hidden clusters, but only the observed values are shown.


In [ ]:
# Hidden cluster labels
z = rng.choice([0, 1], size=300, p=[0.6, 0.4])

# Observed data depends on hidden cluster
x = np.where(
    z == 0,
    rng.normal(-1.0, 0.5, size=300),
    rng.normal(2.0, 0.7, size=300)
)

plt.figure(figsize=(7, 4))
plt.hist(x, bins=30, alpha=0.7, density=True)
plt.title("Observed data from two hidden groups")
plt.xlabel("x")
plt.ylabel("density")
plt.show()

print("The histogram is generated by a mixture of two hidden clusters.")
print("The labels z are not observed during learning.")


## 10. Expectation-Maximization (EM) — Bernoulli Mixture Example

EM is used when part of the data-generating structure is hidden.  
In this example:
- each binary observation is generated by one of two hidden Bernoulli sources,
- the source identity is not observed,
- EM alternates between soft assignment (**E-step**) and parameter re-estimation (**M-step**).


In [ ]:
# Generate data from a mixture of two Bernoulli sources
true_pi = 0.6        # P(component 1)
true_theta1 = 0.8    # P(x=1 | component 1)
true_theta2 = 0.3    # P(x=1 | component 2)

n = 200
z = rng.binomial(1, true_pi, size=n)
x = np.array([
    rng.binomial(1, true_theta1 if zi == 1 else true_theta2)
    for zi in z
])

# Initialize parameters
pi = 0.5
theta1 = 0.6
theta2 = 0.4

def bernoulli_p(xi, theta):
    return theta if xi == 1 else (1 - theta)

log_likelihood_history = []

for iteration in range(20):
    # E-step: responsibilities gamma_i = P(z_i=1 | x_i)
    gamma = np.array([
        pi * bernoulli_p(xi, theta1) /
        (pi * bernoulli_p(xi, theta1) + (1 - pi) * bernoulli_p(xi, theta2))
        for xi in x
    ])

    # M-step
    pi = np.mean(gamma)
    theta1 = np.sum(gamma * x) / np.sum(gamma)
    theta2 = np.sum((1 - gamma) * x) / np.sum(1 - gamma)

    # Log-likelihood
    ll = np.sum([
        np.log(pi * bernoulli_p(xi, theta1) + (1 - pi) * bernoulli_p(xi, theta2))
        for xi in x
    ])
    log_likelihood_history.append(ll)

print(f"Estimated pi      = {pi:.4f}")
print(f"Estimated theta1  = {theta1:.4f}")
print(f"Estimated theta2  = {theta2:.4f}")


### EM log-likelihood over iterations

A useful property of EM is that the data log-likelihood typically increases monotonically until convergence to a local optimum.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(log_likelihood_history, marker="o")
plt.title("EM log-likelihood over iterations")
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()


## 11. Gaussian Mixture Model with EM

A classical application of EM is learning a **mixture of Gaussians**.
The example below implements a simple **1D Gaussian mixture model** from scratch.


In [ ]:
# Generate synthetic data from two 1D Gaussians
data = np.concatenate([
    rng.normal(-2.0, 0.6, size=150),
    rng.normal(2.0, 0.8, size=200)
])

# Initial parameters
w1, w2 = 0.5, 0.5
mu1, mu2 = -1.0, 1.0
sigma1, sigma2 = 1.0, 1.0

def gaussian_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

ll_history = []

for _ in range(30):
    # E-step
    r1 = w1 * gaussian_pdf(data, mu1, sigma1)
    r2 = w2 * gaussian_pdf(data, mu2, sigma2)
    total = r1 + r2
    gamma1 = r1 / total
    gamma2 = r2 / total

    # M-step
    N1 = np.sum(gamma1)
    N2 = np.sum(gamma2)

    w1 = N1 / len(data)
    w2 = N2 / len(data)

    mu1 = np.sum(gamma1 * data) / N1
    mu2 = np.sum(gamma2 * data) / N2

    sigma1 = np.sqrt(np.sum(gamma1 * (data - mu1) ** 2) / N1)
    sigma2 = np.sqrt(np.sum(gamma2 * (data - mu2) ** 2) / N2)

    ll = np.sum(np.log(
        w1 * gaussian_pdf(data, mu1, sigma1) +
        w2 * gaussian_pdf(data, mu2, sigma2)
    ))
    ll_history.append(ll)

print(f"w1={w1:.4f}, mu1={mu1:.4f}, sigma1={sigma1:.4f}")
print(f"w2={w2:.4f}, mu2={mu2:.4f}, sigma2={sigma2:.4f}")


### Visualizing the learned Gaussian mixture

This section plots:
- the learned component densities,
- the final mixture density,
- the EM log-likelihood trajectory.


In [ ]:
grid = np.linspace(np.min(data) - 1, np.max(data) + 1, 500)
pdf1 = w1 * gaussian_pdf(grid, mu1, sigma1)
pdf2 = w2 * gaussian_pdf(grid, mu2, sigma2)
pdf_total = pdf1 + pdf2

plt.figure(figsize=(8, 4))
plt.hist(data, bins=30, density=True, alpha=0.4, label="data")
plt.plot(grid, pdf1, label="component 1")
plt.plot(grid, pdf2, label="component 2")
plt.plot(grid, pdf_total, linewidth=2, label="mixture")
plt.title("Gaussian mixture learned by EM")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(ll_history)
plt.title("GMM EM log-likelihood")
plt.xlabel("Iteration")
plt.ylabel("Log-likelihood")
plt.show()


## 12. Hidden Markov Models — Forward Algorithm

The chapter also discusses learning in hidden Markov models (HMMs).  
Before full learning, students should understand the inference mechanics.

The next cell implements a simple **forward pass** for a two-state HMM.


In [ ]:
states = ["Rainy", "Sunny"]
observations = ["walk", "shop", "clean"]

state_to_idx = {s: i for i, s in enumerate(states)}
obs_to_idx = {o: i for i, o in enumerate(observations)}

# Initial probabilities
pi = np.array([0.6, 0.4])

# Transition matrix A[i, j] = P(X_t+1=j | X_t=i)
A = np.array([
    [0.7, 0.3],
    [0.4, 0.6]
])

# Emission matrix B[i, k] = P(obs_k | state_i)
B = np.array([
    [0.1, 0.4, 0.5],  # Rainy
    [0.6, 0.3, 0.1]   # Sunny
])

sequence = ["walk", "shop", "clean"]
seq_idx = [obs_to_idx[o] for o in sequence]

# Forward algorithm
alpha = np.zeros((len(sequence), len(states)))

alpha[0] = pi * B[:, seq_idx[0]]

for t in range(1, len(sequence)):
    for j in range(len(states)):
        alpha[t, j] = B[j, seq_idx[t]] * np.sum(alpha[t - 1] * A[:, j])

print("Forward probabilities:")
print(alpha)

prob_sequence = np.sum(alpha[-1])
print(f"\nP(observation sequence) = {prob_sequence:.6f}")


### Viterbi decoding

This cell computes the **most likely hidden state sequence** for the same observation sequence.


In [ ]:
T = len(sequence)
N = len(states)

delta = np.zeros((T, N))
psi = np.zeros((T, N), dtype=int)

delta[0] = pi * B[:, seq_idx[0]]

for t in range(1, T):
    for j in range(N):
        probs = delta[t - 1] * A[:, j]
        psi[t, j] = np.argmax(probs)
        delta[t, j] = np.max(probs) * B[j, seq_idx[t]]

path = np.zeros(T, dtype=int)
path[-1] = np.argmax(delta[-1])

for t in range(T - 2, -1, -1):
    path[t] = psi[t + 1, path[t + 1]]

decoded_states = [states[i] for i in path]
print("Most likely hidden states:", decoded_states)


## 13. Summary

This final cell prints a compact summary of the main ideas represented in the notebook.


In [ ]:
summary = {
    "Bayesian learning": "Uses full posterior over hypotheses",
    "MAP learning": "Chooses most probable posterior hypothesis",
    "Maximum likelihood": "Chooses hypothesis maximizing data likelihood",
    "Naive Bayes": "Simple generative classifier with conditional independence assumption",
    "EM": "Alternates hidden-variable expectation and parameter maximization",
    "GMM": "Mixture model commonly learned with EM",
    "HMM": "Sequential latent-state model"
}

for k, v in summary.items():
    print(f"{k:20s} -> {v}")
